# ML Data Preparation - Build and Export the Master Feature Table

Run this **once**. It produces a single wide table (all samples x all
possible features, cleanly numeric, zero-filled where appropriate) plus a
handful of column-list files. The four modelling notebooks (`ml_01`-`ml_04`)
each load this table and just select which columns to use - none of them
touch raw ConQuR/pathway/metadata files directly, so the data-cleaning logic
only exists in one place.

**This version reflects the actual project layout**: ConQuR outputs one
already-merged PD+AD genus table (not two separate tables), significant-genera
lists are CSVs with a `taxon`/`robust_sig` structure (not plain text lists),
and PD/AD pathway tables use different sample-ID schemes and HUMAnN3's PD
table is raw (unrolled, stratified) output rather than a clean pathway-only
matrix. All of that is handled explicitly below rather than assumed away.

**Column naming convention in the exported table:**
- `taxon__<genus>` - all genus-level abundance columns (ConQuR-corrected)
- `pathway__<pathway_id>` - all pathway abundance columns
- `clinical__<var>` - cleaned, numeric clinical variables
- `disease_group`, `study_id`, `platform` - reference columns only,
  never features. `disease_group` is the target, `study_id` is the LOSO
  grouping variable, and `platform` is near-perfectly collinear
  with `disease_group` in this mixed-platform design - including it as a
  feature would just hand the model a shortcut to detect platform instead of
  biology.

Also exported: `significant_taxa_columns.txt` (the subset of `taxon__*`
columns that were ANCOM-BC2-significant in either arm and actually present in
the ConQuR table), `all_taxa_columns.txt`, `clinical_columns.txt`,
`pathway_columns.txt`, and a `feature_manifest.json` recording what was built
and from which files, for reproducibility.


In [19]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 50)


## Step 1.5 - CLR transformation helper

Your original schedule specified CLR (`log(x / geometric_mean(x))`, pseudocount
0.5) applied before merging platforms. `cx_02` deliberately fed ConQuR raw
counts instead - ConQuR's batch-correction model needs count-like input, not
already log-ratio-transformed values - so CLR was always meant to happen
**after** ConQuR, not instead of it. That step was never added. This adds it
here, at the one place downstream where all consumers (`ml_01`-`ml_04`) get
their features from.

**Why this is safe to do post-union (after zero-filling platform-exclusive
taxa/pathways), contrary to the original "separately per platform" plan:**
CLR is a purely row-wise operation - each sample's CLR value only ever
depends on that sample's own feature vector, never on any other row. Whether
you CLR each platform's table separately before combining, or CLR the final
combined table directly, gives mathematically identical results. The
platform-exclusive zero-padding doesn't distort this either: every sample of
a given platform shares the exact same set of zero-padded columns, so it
shifts every same-platform sample's geometric mean by an identical constant
- it doesn't add sample-specific noise.

**Why the pseudocount must be added on the RAW scale, not after any prior
rescaling (e.g. TSS/relative abundance):** `log(x + pseudocount)` is only
scale-invariant to a prior per-sample rescaling in the zero-pseudocount
limit. With a real pseudocount, the same constant behaves completely
differently depending on the scale of `x` - 0.5 is negligible against raw
counts in the millions, but overwhelms a TSS-scaled value between 0 and 1.
So: CLR is applied exactly once, directly to the ConQuR-corrected /
CPM-normalised values as they arrive - no separate relative-abundance step
first.


In [20]:
def apply_clr(df: pd.DataFrame, pseudocount: float = 0.5, label: str = "features") -> pd.DataFrame:
    """
    Row-wise CLR transform: log(x + pseudocount) - mean(log(x + pseudocount))
    across that row's own columns. Matches the project's originally planned
    pseudocount (0.5), applied directly on the raw/count-like scale the
    values arrive in (ConQuR-corrected taxa, CPM pathway abundances) -
    do NOT TSS-normalise first (see markdown above for why that would
    require rescaling the pseudocount too, for no benefit).

    Clips any negative values to 0 first (defensive - ConQuR corrections can
    occasionally produce small negative values near zero; none were observed
    in this project's ConQuR output as of this writing, but log() of a
    negative number is undefined, so this must not fail silently).
    """
    n_negative = (df < 0).sum().sum()
    if n_negative > 0:
        print(f"WARNING: {n_negative} negative values in {label} before CLR - "
              f"clipping to 0 (see apply_clr docstring).")
        df = df.clip(lower=0)

    log_df = np.log(df + pseudocount)
    clr_df = log_df.sub(log_df.mean(axis=1), axis=0)
    print(f"CLR-transformed {label}: {clr_df.shape[1]} columns, "
          f"value range [{clr_df.values.min():.3f}, {clr_df.values.max():.3f}] "
          f"(pseudocount={pseudocount})")
    return clr_df


## Step 1 - Paths (edit all of these)

In [21]:
# ---- EDIT THESE PATHS ----
CONQUR_PATH         = "/rds/projects/e/elhamsak-pd-thesis/cross_platform/conqur_corrected_table.tsv"
METADATA_PATH       = "/rds/projects/e/elhamsak-pd-thesis/cross_platform/master_metadata.csv"
SIG_GENERA_PD_PATH  = "/rds/projects/e/elhamsak-pd-thesis/cross_platform/differential_abundance/pd_specific_genera.csv"
SIG_GENERA_AD_PATH  = "/rds/projects/e/elhamsak-pd-thesis/cross_platform/differential_abundance/ad_specific_genera.csv"
# Pathway abundances (normalised to CPM for machine learning)
PD_PATHWAY_PATH     = "/rds/projects/e/elhamsak-pd-thesis/pd_humann3_merged/pd_pathabundance_cpm.tsv"
AD_PATHWAY_PATH     = "/rds/projects/e/elhamsak-ad-thesis/ad_picrust2_merged/ad_pathabundance_cpm.tsv"

OUTPUT_DIR = Path("/rds/projects/e/elhamsak-ad-thesis/ml/ml_feature_table")
# ---------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Step 2 - Load raw inputs

ConQuR here outputs a **single already-merged** PD+AD genus table indexed by
`unique_sample_id`, with a few text reference columns (`study_id`,
`disease_group`, `platform`) appended by the R export step. Those are dropped
here since they live in `metadata` and would otherwise break the "100%
numeric" assumption the rest of this notebook relies on.


In [22]:
# Load without assuming the first column is the index (write_tsv doesn't write row names)
taxa_raw = pd.read_csv(CONQUR_PATH, sep="\t")
taxa_raw = taxa_raw.set_index("unique_sample_id")

# Drop the text columns appended by R so this matrix is 100% numeric features
cols_to_drop = ["study_id", "disease_group", "platform"]
taxa_raw = taxa_raw.drop(columns=[c for c in cols_to_drop if c in taxa_raw.columns])

metadata = pd.read_csv(METADATA_PATH, sep=",")

# Exclude ueda2021 (excluded upstream from the main AD-vs-HC analyses;
# was still flowing into the ML feature table via master_metadata.csv)
n_before = len(metadata)
metadata = metadata[metadata["study_id"] != "ueda2021"].reset_index(drop=True)
print(f"Excluded ueda2021: {n_before - len(metadata)} rows removed, {len(metadata)} remaining")

print("Unified taxa table (features only):", taxa_raw.shape)
print("Metadata:", metadata.shape)
print("Metadata columns:", list(metadata.columns))


Excluded ueda2021: 7 rows removed, 1887 remaining
Unified taxa table (features only): (1831, 283)
Metadata: (1887, 357)
Metadata columns: ['study_id', 'disease_arm', 'platform', 'amplicon_region', 'location', 'sample_id', 'unique_sample_id', 'run_accession', 'disease_group', 'age', 'sex', 'bmi', 'has_demographics', 'antibiotics', 'antibiotic_free_period', 'smoker', 'updrsiii', 'pd_severity_z', 'moca', 'probiotics', 'bristol_stool_scale', 'race', 'constipation', 'age_of_onset', 'disease_duration', 'hy_stage', 'years_of_education', 'drugs', 'n_runs', 'sample_alias', 'sample_title', 'instrument_platform', 'instrument_model', 'library_layout', 'library_strategy', 'experiment_alias', 'nummer', 'coeruloplasmin_g_l', 'gesamtbilirubin_mg_dl', 'diet', 'birth', 'read_count', 'base_count', 'gsrs', 'first_public', 'id', 'donor_id', 'collection_timestamp', 'description', 'donor_group', 'pd', 'paired', 'host_age', 'host_body_mass_index', 'tube_id', 'laxatives', 'statins', 'proton_pump_inhibitors', '

## Step 3 - Data audit: is everything actually numeric, and what's missing?

This is the check to run before assuming anything is ready for `sklearn`.
`RandomForestClassifier` will error on any non-numeric column and cannot
handle `NaN` at all, so both need to be resolved explicitly, not assumed away.

Note: the `select_dtypes(include="object")` line below prints a harmless
`Pandas4Warning` deprecation notice on newer pandas versions (string columns
are now `StringDtype`, not `object`) - it still runs correctly, so this can be
ignored for now.


In [23]:
print("=== Metadata dtypes ===")
print(metadata.dtypes)
print()

print("=== Metadata missing value counts ===")
print(metadata.isna().sum())
print()

print("=== Unique values for each non-numeric metadata column ===")
for col in metadata.select_dtypes(include="object").columns:
    uniques = metadata[col].unique()
    print(f"\n{col}: {uniques[:10]}{' ...' if len(uniques) > 10 else ''}")
    print(f"  ({metadata[col].nunique()} unique values total)")

print()
print("=== Taxa table numeric check ===")
non_numeric_cols = taxa_raw.select_dtypes(exclude="number").columns.tolist()
print(f"Taxa table: {len(non_numeric_cols)} non-numeric columns "
      f"(should be 0) -> {non_numeric_cols}")
print(f"Taxa table value range: min={taxa_raw.values.min():.4f}, max={taxa_raw.values.max():.4f}")
print(f"Taxa table NaN count: {taxa_raw.isna().sum().sum()}")


=== Metadata dtypes ===
study_id                            str
disease_arm                         str
platform                            str
amplicon_region                     str
location                            str
                                 ...   
apoe_genotype                       str
s_num                           float64
group                               str
mmse                            float64
boktor_constipation_severity    float64
Length: 357, dtype: object

=== Metadata missing value counts ===
study_id                           0
disease_arm                        0
platform                           0
amplicon_region                 1222
location                           0
                                ... 
apoe_genotype                   1832
s_num                           1850
group                           1850
mmse                            1850
boktor_constipation_severity    1855
Length: 357, dtype: int64

=== Unique values for each non-numer

/tmp/ipykernel_3118841/679555900.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in metadata.select_dtypes(include="object").columns:


Read the output above before continuing. In particular:

- If the taxa value range looks like roughly 0-1 (or sums close to 1 per
  sample), that's relative abundance. If it's centred around 0 with negative
  values, that's already CLR-transformed. If it's large positive integers
  (thousands+), that's raw or batch-corrected **counts** - fine for Random
  Forest either way (tree splits don't need feature scaling), but know which
  one you actually have for the Methods write-up, and double check whether
  the CLR-normalisation step you planned actually ran upstream of this table.
- If the taxa table has non-numeric columns or non-zero NaN counts, stop here
  and check the ConQuR output before proceeding - this notebook assumes
  ConQuR's output is already clean.
- Note which metadata columns are text (`sex`, `study_id`, `disease_group`,
  `platform`, and many others) - the ones actually used downstream (`sex`,
  clinical variables) are handled explicitly in Step 6, not silently coerced.


## Step 4 - Taxa columns (ConQuR already merged PD+AD, nothing to union)

The original two-cohort version of this notebook unioned separate PD and AD
tables here. That's not needed anymore - ConQuR's output already covers both
diseases in one table - so this step is now just a prefix/rename pass.


In [26]:
# --- Step 4 (fixed): explode multi-run samples before building the fallback key ---
taxa_X = taxa_raw.copy()

direct_ids = set(metadata["unique_sample_id"].astype(str))

meta_ra = metadata.copy()
meta_ra["run_accession"] = meta_ra["run_accession"].astype(str)
meta_ra_expanded = meta_ra.assign(
    run_accession=meta_ra["run_accession"].str.split(";")
).explode("run_accession")
meta_ra_expanded["run_accession"] = meta_ra_expanded["run_accession"].str.strip()
meta_ra_expanded["_fallback_key"] = (
    meta_ra_expanded["study_id"].astype(str) + "_" + meta_ra_expanded["run_accession"]
)
fallback_map = dict(zip(meta_ra_expanded["_fallback_key"], meta_ra_expanded["unique_sample_id"].astype(str)))

def resolve_id(raw_id):
    raw_id = str(raw_id)
    if raw_id in direct_ids:
        return raw_id
    return fallback_map.get(raw_id)

resolved_index = [resolve_id(i) for i in taxa_X.index]
n_unmatched = sum(r is None for r in resolved_index)
print(f"taxa_X: unmatched after two-stage resolution: {n_unmatched} / {len(taxa_X)}")

taxa_X.index = resolved_index
taxa_X = taxa_X[taxa_X.index.notna()].copy()

# --- NEW: Apply CLR before adding column prefixes ---
taxa_clr = apply_clr(taxa_X, pseudocount=0.5, label="taxa (ConQuR-corrected)")
taxa_X = taxa_clr.copy()

taxa_X.columns = [f"taxon__{c}" for c in taxa_X.columns]

print("taxa_X shape after resolution & CLR:", taxa_X.shape)

taxa_X: unmatched after two-stage resolution: 7 / 1831
CLR-transformed taxa (ConQuR-corrected): 283 columns, value range [-8.137, 16.006] (pseudocount=0.5)
taxa_X shape after resolution & CLR: (1824, 283)


## Step 5 - Significant-genera list (for filtering in ml_02/03/04)

Significant-genera files are CSVs from ANCOM-BC2, not plain newline-separated
lists - each has `taxon`, `lfc`, `q`, `diff_sig`, `passed_ss`, `robust_sig`
columns. Only rows where `robust_sig == TRUE` are kept as "significant" here.
Any significant genus that doesn't actually exist in the ConQuR table (e.g. it
didn't survive prevalence filtering during batch correction) is dropped, with
a warning, rather than causing a `KeyError` several notebooks downstream.


In [27]:
def load_gene_list(path, sig_col="robust_sig"):
    df = pd.read_csv(path)
    df[sig_col] = df[sig_col].astype(str).str.upper() == "TRUE"
    return sorted(set(df.loc[df[sig_col], "taxon"]))

taxa_cols = set(taxa_raw.columns)

sig_pd = load_gene_list(SIG_GENERA_PD_PATH)
sig_ad = load_gene_list(SIG_GENERA_AD_PATH)
sig_union = sorted(set(sig_pd) | set(sig_ad))

missing_from_table = [g for g in sig_union if g not in taxa_cols]

print(f"PD-significant genera: {len(sig_pd)}")
print(f"AD-significant genera: {len(sig_ad)}")
print(f"Union: {len(sig_union)}")
print(f"Significant genera missing from unified table ({len(missing_from_table)}): {missing_from_table}")

if missing_from_table:
    print(f"Dropping {len(missing_from_table)} significant genus/genera not present "
          f"in the taxa table (note these in the Methods/limitations section): {missing_from_table}")
sig_union = [g for g in sig_union if g in taxa_cols]

significant_taxa_columns = [f"taxon__{g}" for g in sig_union]
print(f"Final significant_taxa_columns after dropping missing: {len(significant_taxa_columns)}")


PD-significant genera: 32
AD-significant genera: 12
Union: 44
Significant genera missing from unified table (1): ['g__Granulicatella']
Dropping 1 significant genus/genera not present in the taxa table (note these in the Methods/limitations section): ['g__Granulicatella']
Final significant_taxa_columns after dropping missing: 43


## Step 6 - Clinical metadata: encode, audit missingness, impute

Default clinical set is `age`, `sex`, `bmi` only - variables recorded for
all three disease groups (note: lowercase `bmi` in this metadata, not `BMI`;
there is also a separate `host_body_mass_index` column - confirm that isn't
the one you actually want before relying on `bmi`). `updrsiii`/`moca`/`mmse`
are excluded by default: these instruments aren't administered consistently
across all three disease groups, so their missingness pattern alone would let
a model "detect PD" or "detect AD" just by noticing which instrument was
recorded - a tautology, not a biomarker finding.

`meta_idx` is indexed on `unique_sample_id`, not the short `sample_id` column
- `sample_id` values like `CO-01` repeat across different studies and are not
safe to join on directly.


In [28]:
CLINICAL_VARS = ["age", "sex", "bmi"]   # edit if your column names differ

missing_cols = [c for c in CLINICAL_VARS if c not in metadata.columns]
if missing_cols:
    raise ValueError(f"These clinical columns are not in your metadata: {missing_cols}. "
                      f"Check spelling against: {list(metadata.columns)}")

meta_idx = metadata.set_index("unique_sample_id")
clinical_df = meta_idx[CLINICAL_VARS].copy()

# 'sex' arrives as text ('male'/'female'); map to numeric explicitly rather
# than relying on a dtype check (StringDtype columns won't match `== object`).
clinical_df["sex"] = (
    clinical_df["sex"].astype(str).str.lower().str.strip().map({"male": 0, "female": 1})
)
n_unmapped = clinical_df["sex"].isna().sum() - meta_idx["sex"].isna().sum()
if n_unmapped > 0:
    print(f"WARNING: {n_unmapped} 'sex' values didn't map to male/female - "
          f"check raw values: {meta_idx['sex'].unique()}")

print("Missingness by disease group (before imputation):")
print(clinical_df.join(meta_idx["disease_group"]).groupby("disease_group").apply(lambda d: d.isna().mean()))

for col in CLINICAL_VARS:
    if clinical_df[col].isna().any():
        clinical_df[f"{col}_was_missing"] = clinical_df[col].isna().astype(int)
        clinical_df[col] = clinical_df[col].fillna(clinical_df[col].median())

clinical_df.columns = [f"clinical__{c}" for c in clinical_df.columns]
clinical_columns = clinical_df.columns.tolist()
print("\nFinal clinical feature columns:", clinical_columns)


Missingness by disease group (before imputation):
                    age       sex       bmi
disease_group                              
AD             0.800633  0.800633  0.803797
HC             0.355828  0.354601  0.403681
PD             0.001323  0.041005  0.175926

Final clinical feature columns: ['clinical__age', 'clinical__sex', 'clinical__bmi', 'clinical__age_was_missing', 'clinical__sex_was_missing', 'clinical__bmi_was_missing']


### Optional - adding other clinical instruments anyway (read the warning above first)

In [29]:
# OPTIONAL_CLINICAL_VARS = ["mmse", "updrsiii"]   # only administered in some disease groups
# for col in OPTIONAL_CLINICAL_VARS:
#     was_missing = meta_idx[col].isna().astype(int)
#     clinical_df[f"clinical__{col}_was_missing"] = was_missing.values
#     clinical_df[f"clinical__{col}"] = meta_idx[col].fillna(0).values   # 0 = "not applicable" - document this choice
# clinical_columns = clinical_df.columns.tolist()


## Step 7 - Pathway abundances: filter, harmonise IDs, map samples, union

Three separate problems needed solving here, not just an orientation guess:

1. **PD's HUMAnN3 table is raw, unrolled output.** Its index mixes
   `UNMAPPED`/`UNINTEGRATED` totals, species-stratified sub-rows
   (`UNINTEGRATED|s__Species...`), and the real unstratified pathway rows we
   actually want. Only the unstratified rows are kept.
2. **PD pathway IDs carry a trailing description** (`'1CMET2-PWY: folate
   transformations III (E. coli)'`) that AD's PICRUSt2 IDs don't have
   (`'1CMET2-PWY'`) - stripped so the two ID sets can actually match. Once
   stripped, there's genuine, substantial overlap (161 shared pathways) -
   this is standard MetaCyc-based output on both sides, not a naming-scheme
   mismatch as it first appeared.
3. **Sample IDs on the columns are platform-specific and don't match
   `unique_sample_id` directly** - PD uses `<sample_id>_concat_Abundance`,
   AD uses run accessions matched via `metadata["run_accession"]` (which can
   contain multiple semicolon-joined runs per sample and is expanded
   accordingly). Both are mapped explicitly before anything is joined.


In [30]:
pd_path_raw = pd.read_csv(PD_PATHWAY_PATH, sep="\t", index_col=0)
ad_path_raw = pd.read_csv(AD_PATHWAY_PATH, sep="\t", index_col=0)

print("PD pathway table (raw, as read):", pd_path_raw.shape)
print("AD pathway table (raw, as read):", ad_path_raw.shape)

# Keep only unstratified pathway rows, and strip the trailing ': description'
# HUMAnN3 appends to each pathway ID.
pd_path_unstrat = pd_path_raw[
    (~pd_path_raw.index.str.contains(r"\|", regex=True)) &
    (~pd_path_raw.index.isin(["UNMAPPED", "UNINTEGRATED"]))
].copy()
pd_path_unstrat.index = pd_path_unstrat.index.str.split(":", n=1).str[0].str.strip()

print(f"PD table: {pd_path_raw.shape[0]} raw rows -> {pd_path_unstrat.shape[0]} "
      f"unstratified pathway rows after dropping UNMAPPED/UNINTEGRATED/species-stratified rows")
print("PD unstratified pathway index sample:", pd_path_unstrat.index[:10].tolist())

real_overlap = set(pd_path_unstrat.index) & set(ad_path_raw.index)
print(f"\nReal shared pathway IDs (PD unstratified vs AD): {len(real_overlap)}")
print("Example shared IDs:", sorted(real_overlap)[:10])


PD pathway table (raw, as read): (1920, 1235)
AD pathway table (raw, as read): (414, 617)
PD table: 1920 raw rows -> 206 unstratified pathway rows after dropping UNMAPPED/UNINTEGRATED/species-stratified rows
PD unstratified pathway index sample: ['1CMET2-PWY', 'ANAEROFRUCAT-PWY', 'ANAGLYCOLYSIS-PWY', 'ARGININE-SYN4-PWY', 'ARGSYN-PWY', 'ARGSYNBSUB-PWY', 'ARO-PWY', 'BIOTIN-BIOSYNTHESIS-PWY', 'BRANCHED-CHAIN-AA-SYN-PWY', 'CALVIN-PWY']

Real shared pathway IDs (PD unstratified vs AD): 161
Example shared IDs: ['1CMET2-PWY', 'ANAEROFRUCAT-PWY', 'ANAGLYCOLYSIS-PWY', 'ARGSYN-PWY', 'ARGSYNBSUB-PWY', 'ARO-PWY', 'BIOTIN-BIOSYNTHESIS-PWY', 'BRANCHED-CHAIN-AA-SYN-PWY', 'CALVIN-PWY', 'COA-PWY']


In [31]:
pd_meta = metadata[metadata["disease_arm"] == "PD"].copy()
pd_path_stripped_index = pd_path_unstrat.columns.str.replace("_concat_Abundance", "", regex=False)

# Stage 1: sample_id-based match (covers bedarf2017)
sample_to_unique_pd = pd_meta.drop_duplicates("sample_id").set_index("sample_id")["unique_sample_id"]

# Stage 2: run_accession-based match (covers the other PD studies), same
# exploded-join pattern used for AD, restricted to PD-arm metadata.
pd_meta_ra = pd_meta.copy()
pd_meta_ra["run_accession"] = pd_meta_ra["run_accession"].astype(str)
pd_run_expanded = pd_meta_ra.assign(
    run_accession=pd_meta_ra["run_accession"].str.split(";")
).explode("run_accession")
pd_run_expanded["run_accession"] = pd_run_expanded["run_accession"].str.strip()

dup_pd_runs = pd_run_expanded["run_accession"][pd_run_expanded["run_accession"].duplicated()].unique()
print(f"Duplicate run_accession within PD metadata: {len(dup_pd_runs)}", dup_pd_runs[:10] if len(dup_pd_runs) else "")

run_to_unique_pd = pd_run_expanded.drop_duplicates("run_accession").set_index("run_accession")["unique_sample_id"]

# Try sample_id first, fall back to run_accession
mapped_via_sample_id = pd_path_stripped_index.map(sample_to_unique_pd)
mapped_via_run = pd_path_stripped_index.map(run_to_unique_pd)
final_mapped = mapped_via_sample_id.where(mapped_via_sample_id.notna(), mapped_via_run)

n_still_unmapped = final_mapped.isna().sum()
print(f"PD pathway samples still unmapped after both stages: {n_still_unmapped} / {len(final_mapped)}")
if n_still_unmapped > 0:
    still_unmapped_ids = pd_path_stripped_index[final_mapped.isna()]
    print("Examples still unmapped:", still_unmapped_ids[:10].tolist())

pd_path_T = pd_path_unstrat.T
pd_path_T.index = final_mapped
pd_path_T.index.name = "sample_id"
pd_path_T = pd_path_T[pd_path_T.index.notna()]

Duplicate run_accession within PD metadata: 0 
PD pathway samples still unmapped after both stages: 21 / 1235
Examples still unmapped: ['ERR9830085', 'ERR9830110', 'ERR9830162', 'ERR9830165', 'ERR9830171', 'ERR9830195', 'ERR9830233', 'ERR9830235', 'ERR9830260', 'ERR9830267']


In [32]:
# --- AD sample-ID mapping: run_accession (possibly semicolon-joined) -> unique_sample_id ---
ad_meta = metadata[metadata["disease_arm"] == "AD"].copy()
ad_meta["run_accession"] = ad_meta["run_accession"].astype(str)
ad_run_expanded = ad_meta.assign(
    run_accession=ad_meta["run_accession"].str.split(";")
).explode("run_accession")
ad_run_expanded["run_accession"] = ad_run_expanded["run_accession"].str.strip()

dup_runs = ad_run_expanded["run_accession"][ad_run_expanded["run_accession"].duplicated()].unique()
print(f"Duplicate run_accession within AD metadata: {len(dup_runs)}", dup_runs[:10] if len(dup_runs) else "")

run_to_unique = ad_run_expanded.drop_duplicates("run_accession").set_index("run_accession")["unique_sample_id"]

ad_path_T = ad_path_raw.T
ad_path_T.index = ad_path_T.index.map(run_to_unique)
n_unmapped_ad = ad_path_T.index.isna().sum()
print(f"AD pathway samples that failed to map to unique_sample_id: {n_unmapped_ad} / {len(ad_path_T)}")
ad_path_T = ad_path_T[ad_path_T.index.notna()]


Duplicate run_accession within AD metadata: 0 
AD pathway samples that failed to map to unique_sample_id: 7 / 617


In [33]:
# # --- Assemble: union of pathway columns, zero-filled, both tables now on unique_sample_id ---
# pd_path = pd_path_T
# pd_path.index.name = "sample_id"

# ad_path = ad_path_T
# ad_path.index.name = "sample_id"

# pd_path_cols = set(pd_path.columns)
# ad_path_cols = set(ad_path.columns)
# pathway_union = sorted(pd_path_cols | ad_path_cols)
# shared_pathways = sorted(pd_path_cols & ad_path_cols)

# print(f"Pathways in both tables: {len(shared_pathways)}")
# print(f"Pathways only in PD table: {len(pd_path_cols - ad_path_cols)}")
# print(f"Pathways only in AD table: {len(ad_path_cols - pd_path_cols)}")
# print(f"Total pathway feature set: {len(pathway_union)}")

# if len(shared_pathways) == 0:
#     print("\nWARNING: zero shared pathways - re-check the ID stripping/orientation logic above.")

# pd_path_X = pd_path.reindex(columns=pathway_union, fill_value=0.0)
# ad_path_X = ad_path.reindex(columns=pathway_union, fill_value=0.0)
# pathway_X = pd.concat([pd_path_X, ad_path_X], axis=0)
# pathway_X.index.name = "sample_id"
# pathway_X.columns = [f"pathway__{c}" for c in pathway_X.columns]
# pathway_columns = pathway_X.columns.tolist()

# print("\nCombined pathway matrix:", pathway_X.shape)


In [34]:
# --- Assemble: union of pathway columns, zero-filled ---
pd_path = pd_path_T
pd_path.index.name = "sample_id"

ad_path = ad_path_T
ad_path.index.name = "sample_id"

pd_path_cols = set(pd_path.columns)
ad_path_cols = set(ad_path.columns)
pathway_union = sorted(pd_path_cols | ad_path_cols)
shared_pathways = sorted(pd_path_cols & ad_path_cols)

print(f"Pathways in both tables: {len(shared_pathways)}")
print(f"Pathways only in PD table: {len(pd_path_cols - ad_path_cols)}")
print(f"Pathways only in AD table: {len(ad_path_cols - pd_path_cols)}")
print(f"Total pathway feature set: {len(pathway_union)}")

pd_path_X = pd_path.reindex(columns=pathway_union, fill_value=0.0)
ad_path_X = ad_path.reindex(columns=pathway_union, fill_value=0.0)
pathway_X_raw = pd.concat([pd_path_X, ad_path_X], axis=0)
pathway_X_raw.index.name = "sample_id"

print("\nCombined pathway matrix (pre-CLR):", pathway_X_raw.shape)

# --- Exclude samples with fully-zero pathway rows ---
all_zero_mask = (pathway_X_raw.sum(axis=1) == 0)
broken_pathway_samples = pathway_X_raw.index[all_zero_mask].tolist()
if broken_pathway_samples:
    print(f"\nWARNING: {len(broken_pathway_samples)} sample(s) have an all-zero pathway row "
          f"- excluding and logging: {broken_pathway_samples}")
    exclusions_log_path = OUTPUT_DIR / "sample_exclusions_log.tsv"
    log_rows = pd.DataFrame({
        "sample_id": broken_pathway_samples,
        "reason": "all_zero_pathway_row_likely_broken_merge",
        "excluded_from": "pathway_columns_only",
        "logged_at": datetime.now().isoformat(),
    })
    log_rows.to_csv(exclusions_log_path, mode="a",
                    header=not exclusions_log_path.exists(), sep="\t", index=False)
    pathway_X_raw = pathway_X_raw.loc[~all_zero_mask]

# --- Apply CLR to Pathways ---
pathway_clr = apply_clr(pathway_X_raw, pseudocount=0.5, label="pathways (CPM-normalised)")
pathway_X = pathway_clr.copy()
pathway_X.columns = [f"pathway__{c}" for c in pathway_X.columns]
pathway_columns = pathway_X.columns.tolist()

print("\nCombined pathway matrix (CLR-transformed):", pathway_X.shape)

Pathways in both tables: 161
Pathways only in PD table: 45
Pathways only in AD table: 253
Total pathway feature set: 459

Combined pathway matrix (pre-CLR): (1824, 459)

CLR-transformed pathways (CPM-normalised): 459 columns, value range [-6.207, 6.767] (pseudocount=0.5)

Combined pathway matrix (CLR-transformed): (1822, 459)


## Step 8 - Assemble and export the master feature table

Joins taxa + clinical + pathway blocks plus the three reference columns
(`disease_group`, `study_id`, `platform`). Rows that fail to join across any
block are flagged, not silently dropped without a warning. All three blocks
(`taxa_X`, `clinical_df`, `pathway_X`, `meta_idx`) are now consistently
indexed on `unique_sample_id`, so this join should no longer silently drop
rows the way it would have with the original `sample_id`-indexed metadata.


In [35]:
master = taxa_X.join(clinical_df, how="left")
master = master.join(pathway_X, how="left")
master = master.join(meta_idx[["disease_group", "study_id", "platform"]], how="inner")

n_dropped = taxa_X.shape[0] - master.shape[0]
if n_dropped > 0:
    print(f"WARNING: {n_dropped} samples dropped during the metadata join - check sample_id formatting.")

# pathway_na_count = master[pathway_columns].isna().sum().sum() if pathway_columns else 0
# if pathway_na_count > 0:
#     print(f"Filling {pathway_na_count} missing pathway values with 0 (samples with no pathway-table match).")
#     master[pathway_columns] = master[pathway_columns].fillna(0.0)

# Drop samples flagged with invalid functional profiles so CLR isn't distorted
if broken_pathway_samples:
    print(f"Dropping {len(broken_pathway_samples)} samples with broken pathway profiles from master table: {broken_pathway_samples}")
    master = master.drop(index=[s for s in broken_pathway_samples if s in master.index])

clinical_na_count = master[clinical_columns].isna().sum().sum()
if clinical_na_count > 0:
    print(f"WARNING: {clinical_na_count} unexpected NaNs remain in clinical columns after imputation in Step 6 - investigate.")

print("\nFinal master table shape:", master.shape)
print(master["disease_group"].value_counts())


Dropping 2 samples with broken pathway profiles from master table: ['boktor2023_tbc_PD.9', 'boktor2023_rumc_PC.66']

Final master table shape: (1822, 751)
disease_group
HC    755
PD    751
AD    316
Name: count, dtype: int64


In [36]:
master.to_csv(OUTPUT_DIR / "ml_master_feature_table.csv")

with open(OUTPUT_DIR / "all_taxa_columns.txt", "w") as f:
    f.write("\n".join(taxa_X.columns.tolist()))
with open(OUTPUT_DIR / "significant_taxa_columns.txt", "w") as f:
    f.write("\n".join(significant_taxa_columns))
with open(OUTPUT_DIR / "clinical_columns.txt", "w") as f:
    f.write("\n".join(clinical_columns))
with open(OUTPUT_DIR / "pathway_columns.txt", "w") as f:
    f.write("\n".join(pathway_columns))

manifest = {
    "built_at": datetime.now().isoformat(),
    "source_files": {
        "conqur": CONQUR_PATH,
        "metadata": METADATA_PATH,
        "sig_genera_pd": SIG_GENERA_PD_PATH,
        "sig_genera_ad": SIG_GENERA_AD_PATH,
        "pd_pathway": PD_PATHWAY_PATH,
        "ad_pathway": AD_PATHWAY_PATH,
    },
    "n_samples": master.shape[0],
    "n_all_taxa": len(taxa_X.columns),
    "n_significant_taxa": len(significant_taxa_columns),
    "n_clinical_features": len(clinical_columns),
    "n_pathway_features": len(pathway_union),
    "n_shared_pathways_across_platforms": len(shared_pathways),
    "disease_group_counts": master["disease_group"].value_counts().to_dict(),
    "normalisation": {
        "method": "CLR (centered log-ratio)",
        "pseudocount": 0.5,
        "applied_to": ["taxa (post-ConQuR)", "pathways (post-CPM-normalisation)"],
        "applied_separately_per_block": True,
        "note": "Row-wise transform applied directly to count/CPM scale."
    },
    "samples_excluded_broken_pathway_merge": broken_pathway_samples,
}
with open(OUTPUT_DIR / "feature_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Exported to:", OUTPUT_DIR)
print(json.dumps(manifest, indent=2))


Exported to: /rds/projects/e/elhamsak-ad-thesis/ml/ml_feature_table
{
  "built_at": "2026-09-14T07:20:16.447430",
  "source_files": {
    "conqur": "/rds/projects/e/elhamsak-pd-thesis/cross_platform/conqur_corrected_table.tsv",
    "metadata": "/rds/projects/e/elhamsak-pd-thesis/cross_platform/master_metadata.csv",
    "sig_genera_pd": "/rds/projects/e/elhamsak-pd-thesis/cross_platform/differential_abundance/pd_specific_genera.csv",
    "sig_genera_ad": "/rds/projects/e/elhamsak-pd-thesis/cross_platform/differential_abundance/ad_specific_genera.csv",
    "pd_pathway": "/rds/projects/e/elhamsak-pd-thesis/pd_humann3_merged/pd_pathabundance_cpm.tsv",
    "ad_pathway": "/rds/projects/e/elhamsak-ad-thesis/ad_picrust2_merged/ad_pathabundance_cpm.tsv"
  },
  "n_samples": 1822,
  "n_all_taxa": 283,
  "n_significant_taxa": 43,
  "n_clinical_features": 6,
  "n_pathway_features": 459,
  "n_shared_pathways_across_platforms": 161,
  "disease_group_counts": {
    "HC": 755,
    "PD": 751,
    "AD": 

In [37]:
# --- Feature-set manifests: primary (no clinical) vs clinical-inclusive (secondary) ---
primary_feature_columns = [
    c for c in master.columns if c.startswith("taxon__") or c.startswith("pathway__")
]
clinical_inclusive_feature_columns = primary_feature_columns + clinical_columns

print(f"Primary (no-clinical) feature set: {len(primary_feature_columns)} columns")
print(f"Clinical-inclusive feature set: {len(clinical_inclusive_feature_columns)} columns")

with open(OUTPUT_DIR / "primary_feature_columns.txt", "w") as f:
    f.write("\n".join(primary_feature_columns))
with open(OUTPUT_DIR / "clinical_inclusive_feature_columns.txt", "w") as f:
    f.write("\n".join(clinical_inclusive_feature_columns))

Primary (no-clinical) feature set: 742 columns
Clinical-inclusive feature set: 748 columns


In [38]:
manifest["n_primary_features"] = len(primary_feature_columns)
manifest["n_clinical_inclusive_features"] = len(clinical_inclusive_feature_columns)
manifest["clinical_block_caveat"] = (
    "age/sex/bmi excluded from primary model: AD missingness (~80%) is "
    "concentrated in specific AD studies, not spread evenly across samples, "
    "so imputation flags remain partially confounded with study_id even "
    "under LOSO cross-validation. Reported separately as a clinical-inclusive "
    "secondary analysis."
)